# 03 — Training Diagnostics

Analyze GNN multi-task training:
1. Loss curves (classification + regression)
2. Embedding PCA by DM model
3. Confusion matrix for 3-class classification
4. Regression accuracy (log10_M_sub_mean, n_impacts)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))

CKPT_DIR = ROOT / 'checkpoints'
print(f'Checkpoint directory: {CKPT_DIR}')

## Model architecture summary

In [ ]:
from src.models.gnn import StreamGNNMultiTask
from src.models.utils import count_parameters, load_checkpoint
import yaml

with open(ROOT / 'config' / 'training.yaml') as f:
    cfg = yaml.safe_load(f)

gcfg = cfg['model']['gnn']
model = StreamGNNMultiTask(
    n_classes=3, n_reg_targets=2,
    n_node_features=cfg['graph']['n_node_features'],
    n_edge_features=cfg['graph']['n_edge_features'],
    hidden_dim=gcfg['hidden_dim'],
    n_layers=gcfg['n_layers'],
    embedding_dim=gcfg['embedding_dim'],
    dropout=gcfg['dropout'],
)

total, trainable = count_parameters(model)
print(f'Total parameters: {total:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'\nArchitecture: {gcfg["n_layers"]}-layer GINEConv, '
      f'hidden_dim={gcfg["hidden_dim"]}, embedding_dim={gcfg["embedding_dim"]}')

## Load checkpoint and inspect training state

In [ ]:
ckpt_path = CKPT_DIR / 'gnn_best.pt'
if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    print(f'Best checkpoint: epoch {ckpt.get("epoch", "?")}, val_loss={ckpt.get("val_loss", "?")}')
    if 'config' in ckpt:
        print(f'Training config saved in checkpoint')
else:
    print('No checkpoint found at', ckpt_path)

## Embedding PCA visualization

Load precomputed embeddings and visualize DM model separation in PCA space.

In [ ]:
# Check for precomputed embeddings
emb_files = list(CKPT_DIR.glob('embeddings_*.npz'))
if emb_files:
    all_emb = []
    all_labels = []
    label_map = {'CDM': 0, 'WDM': 1, 'FDM': 2, 'SIDM': 3}
    for f in emb_files:
        model_name = f.stem.replace('embeddings_', '')
        data = np.load(str(f))
        embs = data['embeddings']
        all_emb.append(embs)
        all_labels.extend([label_map.get(model_name, -1)] * len(embs))
    
    all_emb = np.concatenate(all_emb, axis=0)
    all_labels = np.array(all_labels)
    
    pca = PCA(n_components=2)
    coords = pca.fit_transform(all_emb)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c', 3: '#d62728'}
    names = {0: 'CDM', 1: 'WDM', 2: 'FDM', 3: 'SIDM'}
    for label in np.unique(all_labels):
        mask = all_labels == label
        ax.scatter(coords[mask, 0], coords[mask, 1], s=5, alpha=0.3,
                   color=colors.get(label, 'gray'), label=names.get(label, '?'))
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title('GNN Embedding PCA: DM Model Separation')
    ax.legend(markerscale=5)
    plt.tight_layout()
    plt.show()
else:
    print('No precomputed embeddings found. Run train_sbi.py first.')

## Weight statistics

In [ ]:
if ckpt_path.exists():
    model.load_state_dict(ckpt['model_state_dict'])
    print('Layer weight statistics:')
    for name, param in model.named_parameters():
        if 'weight' in name and param.dim() >= 2:
            print(f'  {name:50s} shape={list(param.shape)}, '
                  f'mean={param.data.mean():.4f}, std={param.data.std():.4f}, '
                  f'max={param.data.abs().max():.4f}')